In [10]:
import importlib
import pandas as pd

import load_data
import labels
import build_features
import extract_ontology
import build_topology_graph
import build_ontology
import util

importlib.reload(load_data)
importlib.reload(labels)
importlib.reload(build_features)
importlib.reload(extract_ontology)
importlib.reload(build_topology_graph)
importlib.reload(build_ontology)
importlib.reload(util)

from load_data import load_alerts_from_json
from labels import add_labels_to_df
from extract_ontology import topology_edges, host_sets
from build_topology_graph import plot_scenario_topologies
from build_ontology import build_all_scenarios
from util import attacks_per_period_report

### Load data
##### Wazuh + Aminer Alerts (AIT-ADS, .json format) to .csv

In [2]:
output_file = "combined_ait.csv"
dir_path = "../data/ait_ads"

df = load_alerts_from_json(output_file, dir_path)

Opening file ../data/ait_ads/harrison_aminer.json...
Opening file ../data/ait_ads/wardbeck_wazuh.json...
Opening file ../data/ait_ads/wheeler_wazuh.json...
Opening file ../data/ait_ads/shaw_wazuh.json...
Opening file ../data/ait_ads/wilson_aminer.json...
Opening file ../data/ait_ads/fox_aminer.json...
Opening file ../data/ait_ads/santos_wazuh.json...
Opening file ../data/ait_ads/fox_wazuh.json...
Opening file ../data/ait_ads/shaw_aminer.json...
Opening file ../data/ait_ads/wheeler_aminer.json...
Opening file ../data/ait_ads/russellmitchell_aminer.json...
Opening file ../data/ait_ads/harrison_wazuh.json...
Opening file ../data/ait_ads/wilson_wazuh.json...
Opening file ../data/ait_ads/santos_aminer.json...
Opening file ../data/ait_ads/russellmitchell_wazuh.json...
Opening file ../data/ait_ads/wardbeck_aminer.json...
Writing data to combined_ait.csv...

Done.


##### Load .csv to notebook

In [3]:
df = pd.read_csv("../data/ait_ads/combined_ait.csv", low_memory=False)
df.head()

Min timestamp in dataset:  2022-01-14 00:00:01.750000+00:00 
 Max timestamp in dataset:  2022-02-08 23:59:46.130000114+00:00


#### Sanity checks

In [9]:
print("Min timestamp in dataset: ", df["timestamp"].min(), "\nMax timestamp in dataset: ", df["timestamp"].max())
df.columns

Min timestamp in dataset:  2022-01-14 00:00:01.750000+00:00 
Max timestamp in dataset:  2022-02-08 23:59:46.130000114+00:00


Index(['timestamp', 'scenario', 'source', 'category', 'entity', 'raw_log',
       'host_ip', 'host', 'rule_id', 'rule_desc', 'groups', 'groups_raw',
       'groups_str', 'alert_channel', 'decoder', 'decoder_parent', 'location',
       'mitre_ids', 'mitre_tactic', 'mitre_technique', 'username', 'procname',
       'aminer_component_type', 'aminer_training_mode', 'aminer_new_event',
       'srcip', 'dstip', 'srcport', 'dstport', 'proto', 'is_ids_alert',
       'ids_signature', 'ids_category', 'ids_severity', 'data_json',
       'wazuh_level'],
      dtype='object')

## Labelling

### Attack report

In [ ]:
# number of attakcs in the dataset for each scenario
print(df["y"].value_counts())

df[df["y"] == 1]["scenario"].value_counts()
# or
df.groupby("scenario")["y"].sum().sort_values(ascending=False)

overall, by_scn, summary = attacks_per_period_report(
    df,
    period="day",                  # "hour" | "day" | "week"
    scenario_col="scenario",
    tz="UTC",
    week_start="MON",
)

print(summary)
display(overall.head(10))
display(overall[~overall["has_both_classes"]].head(20))

In [ ]:
add_labels_to_df(df)

event_label
benign    2509439
attack     146382
Name: count, dtype: int64
scenario         event_label
fox              benign         441601
                 attack          31503
harrison         benign         561805
                 attack          32143
russellmitchell  benign          39089
                 attack           6455
santos           benign         123766
                 attack           7013
shaw             benign          69009
                 attack           1773
wardbeck         benign          89296
                 attack           1961
wheeler          benign         580288
                 attack          35873
wilson           benign         604585
                 attack          29661
Name: count, dtype: int64
Writing to output file...
Done.


: 

In [ ]:
df.columns

Index(['timestamp', 'category', 'entity', 'raw_log', 'scenario', 'source',
       'host_ip', 'host', 'rule_id', 'rule_desc', 'groups', 'groups_raw',
       'groups_str', 'srcip', 'dstip', 'srcport', 'dstport', 'proto',
       'is_auth_event', 'is_cred_event', 'is_web_event', 'is_cron',
       'is_success', 'is_uid0', 'aminer_component_type',
       'aminer_training_mode', 'aminer_new_event', 'wazuh_level',
       'wazuh_antivirus', 'wazuh_update', 'is_ids_alert', 'ids_signature',
       'ids_category', 'ids_severity', 'exploit_class', 'alert_channel',
       'decoder', 'decoder_parent', 'location', 'mitre_ids', 'mitre_tactic',
       'mitre_technique', 'username', 'procname'],
      dtype='object')

: 

Check start and end times per scenario